# Netflix Recommendation System — Task 1: Data Loading & Preprocessing

This notebook loads the raw Netflix Prize Dataset (or triggers the synthetic data fallback generator), filters it to the top 5,000 most active users and top 2,000 most rated movies, and performs an 80/20 stratified train-test split.

In [1]:
import sys
import os
import pandas as pd
import pickle

# Add parent directory to path to import src modules
sys.path.append(os.path.abspath('..'))
from src import config, data_loader, preprocessor

## 1. Load and Process Dataset

We will call `load_and_preprocess_dataset()` which parses the raw Netflix text files (or generates synthetic Netflix-style data if missing), subsets the data, and saves it as parquet.

In [2]:
ratings_df, movies_df = data_loader.load_and_preprocess_dataset()

Parsing movie titles from C:\Users\vivek parihar\.gemini\antigravity\scratch\netflix-recommendation-system\data\movie_titles.csv...
Pass 1/2: Counting user activity and movie popularity in raw text files...
  Scanning combined_data_1.txt...


  Scanning combined_data_2.txt...


  Scanning combined_data_3.txt...


  Scanning combined_data_4.txt...


Identifying top 5000 active users and top 2000 popular movies...
Pass 2/2: Extracting filtered ratings for top users and movies...
  Parsing ratings from combined_data_1.txt...


...

  Parsing ratings from combined_data_3.txt...


  Parsing ratings from combined_data_4.txt...


Building filtered DataFrame with 5,352,780 ratings...


Saving processed data to parquet: C:\Users\vivek parihar\.gemini\antigravity\scratch\netflix-recommendation-system\data\processed_ratings.parquet...


Data loading and subsetting complete.
Final filtered dataset contains 5,352,780 ratings, 5000 unique users, and 2000 unique movies.


## 2. Verify Data Shapes and Columns

In [3]:
print('Ratings DataFrame Columns:', ratings_df.columns.tolist())
print('Movies DataFrame Columns:', movies_df.columns.tolist())
print(f'Total ratings: {len(ratings_df):,}')
print(f'Unique users: {ratings_df["user_id"].nunique():,}')
print(f'Unique movies: {ratings_df["movie_id"].nunique():,}')

Ratings DataFrame Columns: ['user_id', 'movie_id', 'rating', 'date', 'year', 'movie_title']
Movies DataFrame Columns: ['movie_id', 'year', 'movie_title']
Total ratings: 5,352,780
Unique users: 5,000
Unique movies: 2,000


## 3. Perform Stratified Train-Test Split

We split the data into 80% training and 20% testing, stratified by user, which ensures every user in the test set is present in the train set.

In [4]:
train_df, test_df = preprocessor.split_data_stratified(
    ratings_df, 
    test_size=config.TEST_SIZE, 
    random_state=config.RANDOM_STATE
)

Performing stratified train-test split (test_size=0.2)...


Split complete. Train set: 4,282,211 ratings. Test set: 1,070,569 ratings.


## 4. Map IDs to Contiguous Integer Indices

We fit an `IndexMapper` on the ratings dataset and save it to results/ index_mapper.pkl.

In [5]:
mapper = preprocessor.IndexMapper()
mapper.fit(ratings_df)

print('Mapped Number of Users:', mapper.num_users)
print('Mapped Number of Movies:', mapper.num_movies)

# Save the mapper
mapper_path = os.path.join(config.RESULTS_DIR, 'index_mapper.pkl')
with open(mapper_path, 'wb') as f:
    pickle.dump(mapper, f)
print(f'Saved mapper to {mapper_path}')

Mapped Number of Users: 5000
Mapped Number of Movies: 2000
Saved mapper to C:\Users\vivek parihar\.gemini\antigravity\scratch\netflix-recommendation-system\results\index_mapper.pkl


## 5. Save Splits as Parquet

We save the train-test sets as parquet for downstream notebooks.

In [6]:
train_path = os.path.join(config.DATA_DIR, 'train_ratings.parquet')
test_path = os.path.join(config.DATA_DIR, 'test_ratings.parquet')

train_df.to_parquet(train_path, index=False)
test_df.to_parquet(test_path, index=False)
print(f'Saved train set to {train_path}')
print(f'Saved test set to {test_path}')

Saved train set to C:\Users\vivek parihar\.gemini\antigravity\scratch\netflix-recommendation-system\data\train_ratings.parquet
Saved test set to C:\Users\vivek parihar\.gemini\antigravity\scratch\netflix-recommendation-system\data\test_ratings.parquet
